# 29 — Evidence-gated source-only domain generalization

This notebook tests whether the established lower-back acceleration classifier can be improved without using either frozen external cohort for model selection. The development sources are Felius, Voisard, and Sint Maartenskliniek. RevalExo and NONAN remain unread.

## Predeclared comparison

1. **ERM control:** established Inception encoder, source/class-balanced batches, AdamW.
2. **Deep CORAL:** the same encoder and classifier plus the released mean-and-covariance alignment loss (weight 1.0).
3. **ERM++-style:** the same encoder with classifier warm-up, Adam weight decay, and simple moving-average evaluation. It is labelled *style* because an ImageNet-pretrained time-series encoder is unavailable.
4. **Secondary complementary-error audit:** after neither standalone candidate passed the first gate, deterministic equal-probability ensembles are evaluated from the matched out-of-source participant predictions. This rescue analysis is explicitly exploratory and uses development cohorts only.

The design follows the official [Deep CORAL implementation](https://github.com/VisionLearningGroup/CORAL), the official [ERM++ implementation](https://github.com/lovelyqian/ERMPlusPlus), and the HAROOD wearable domain-generalization benchmark. None establishes universal superiority, so this notebook requires empirical non-inferiority and a material gain.

## Leakage controls and decision rule

- Outer evaluation: leave one entire development source out.
- Inner tuning: participant-disjoint 20% validation inside the two training sources only.
- Epoch choice: 4/8/12/16 using worst-source balanced accuracy, then specificity, Brier score, and earlier stopping.
- Final repetitions: seeds 42, 137, 202, 314, and 515.
- Primary input: lower-back acceleration magnitude only, preserving the original research question.
- Candidate acceptance: predefined worst-source and paired-bootstrap non-inferiority bounds plus a material gain.
- Three-channel confirmation runs only if a standalone method or fixed ensemble passes the lower-back gate.


In [1]:
from pathlib import Path
import json, sys
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models.evidence_gated_domain_generalization import BenchmarkConfig, run

assert torch.cuda.is_available(), 'GPU is required for this benchmark'
module_text = (ROOT / 'src/models/evidence_gated_domain_generalization.py').read_text(encoding='utf-8')
for forbidden in ('revalexo_external_windows', 'nonan_external_windows'):
    assert forbidden not in module_text.lower(), f'Frozen-cohort load reference found: {forbidden}'
print('Project:', ROOT)
print('GPU:', torch.cuda.get_device_name(0))
print('Frozen evaluation cohorts referenced by loader: NONE')


Project: C:\Users\frank\Documents\MR-ICT Review Paper
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
Frozen evaluation cohorts referenced by loader: NONE


In [2]:
lower_back_config = BenchmarkConfig(representation='lower_back_acceleration')
lower_back = run(ROOT, lower_back_config)


Device: cuda | NVIDIA GeForce RTX 5060 Laptop GPU
Representation: lower_back_acceleration | channels=[0]
source                label  
felius_2024           healthy     34
                      stroke     129
sint_maartenskliniek  healthy     20
                      stroke      10
voisard_2025          healthy     72
                      stroke      49
Name: participants, dtype: int64


C:\Users\frank\Documents\MR-ICT Review Paper\src\models\evidence_gated_domain_generalization.py:410: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  validation_history = history_frame.loc[history_frame.get("validation", False).fillna(False)].copy()


Tuned felius_2024 erm: 12 epochs


C:\Users\frank\Documents\MR-ICT Review Paper\src\models\evidence_gated_domain_generalization.py:410: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  validation_history = history_frame.loc[history_frame.get("validation", False).fillna(False)].copy()


Tuned felius_2024 coral: 16 epochs


...

Complete voisard_2025 seed=314 method=ermpp_style


Complete voisard_2025 seed=515 method=erm


Complete voisard_2025 seed=515 method=coral


Complete voisard_2025 seed=515 method=ermpp_style


            method      held_out_source  auroc  brier  balanced_accuracy  specificity  sensitivity  false_positives
             coral          felius_2024 0.8532 0.2047             0.7385       0.7824       0.6946              7.4
             coral sint_maartenskliniek 0.8970 0.1271             0.8050       0.8100       0.8000              3.8
             coral         voisard_2025 0.9243 0.1321             0.8319       0.7250       0.9388             19.8
      ensemble_all          felius_2024 0.8588 0.1551             0.7637       0.7647       0.7628              8.0
      ensemble_all sint_maartenskliniek 0.8950 0.1387             0.8400       0.8000       0.8800              4.0
      ensemble_all         voisard_2025 0.9108 0.1338             0.8384       0.7583       0.9184             17.4
ensemble_erm_coral          felius_2024 0.8556 0.1805             0.7459       0.7647       0.7271              8.0
ensemble_erm_coral sint_maartenskliniek 0.8950 0.1317             0.8250

In [3]:
display(lower_back['tuning'].sort_values(['held_out_source', 'method', 'epoch']))
display(lower_back['summary'].sort_values(['method', 'held_out_source']).round(4))
print(json.dumps(lower_back['decision'], indent=2))


,held_out_source,method,epoch,worst_balanced_accuracy,worst_specificity,worst_sensitivity,worst_auroc,mean_brier,representation
4,felius_2024,coral,4,0.807143,0.714286,0.900000,0.914286,0.092514,lower_back_acceleration
5,felius_2024,coral,8,0.764286,0.928571,0.600000,0.921429,0.063590,lower_back_acceleration
6,felius_2024,coral,12,0.778571,0.857143,0.700000,0.942857,0.058107,lower_back_acceleration
7,felius_2024,coral,16,0.814286,0.928571,0.700000,0.950000,0.063386,lower_back_acceleration
0,felius_2024,erm,4,0.728571,0.857143,0.600000,0.921429,0.066963,lower_back_acceleration
1,felius_2024,erm,8,0.764286,0.928571,0.600000,0.935714,0.062068,lower_back_acceleration
2,felius_2024,erm,12,0.878571,0.857143,0.900000,0.928571,0.058706,lower_back_acceleration
3,felius_2024,erm,16,0.850000,1.000000,0.700000,0.950000,0.051737,lower_back_acceleration
8,felius_2024,ermpp_style,4,0.771429,0.642857,0.900000,0.892857,0.151908,lower_back_acceleration
9,felius_2024,ermpp_style,8,0.771429,0.642857,0.900000,0.900000,0.132253,lower_back_acceleration


,method,held_out_source,auroc,brier,balanced_accuracy,specificity,sensitivity,false_positives
0,coral,felius_2024,0.8532,0.2047,0.7385,0.7824,0.6946,7.4
1,coral,sint_maartenskliniek,0.8970,0.1271,0.8050,0.8100,0.8000,3.8
2,coral,voisard_2025,0.9243,0.1321,0.8319,0.7250,0.9388,19.8
3,ensemble_all,felius_2024,0.8588,0.1551,0.7637,0.7647,0.7628,8.0
4,ensemble_all,sint_maartenskliniek,0.8950,0.1387,0.8400,0.8000,0.8800,4.0
5,ensemble_all,voisard_2025,0.9108,0.1338,0.8384,0.7583,0.9184,17.4
6,ensemble_erm_coral,felius_2024,0.8556,0.1805,0.7459,0.7647,0.7271,8.0
7,ensemble_erm_coral,sint_maartenskliniek,0.8950,0.1317,0.8250,0.8100,0.8400,3.8
8,ensemble_erm_coral,voisard_2025,0.9182,0.1495,0.8137,0.6806,0.9469,23.0
9,ensemble_erm_ermpp,felius_2024,0.8581,0.1394,0.7676,0.7353,0.8000,9.0


{
  "selected_method": "ensemble_all",
  "candidate_decisions": {
    "coral": {
      "accepted": false,
      "noninferior": false,
      "material_gain": true,
      "worst_source_deltas": {
        "balanced_accuracy_delta": -0.020953032375740976,
        "specificity_delta": 0.09444444444444444,
        "sensitivity_delta": -0.07131782945736431,
        "auroc_delta": -0.0020519835841315004,
        "mean_brier_delta": -0.011782791520719038
      },
      "paired_bootstrap": {
        "balanced_accuracy": {
          "mean_delta": 0.008382246955629799,
          "ci_low": -0.03144447423672311,
          "ci_high": 0.05277676277229233
        },
        "specificity": {
          "mean_delta": 0.057952069716775606,
          "ci_low": -0.024968137254901945,
          "ci_high": 0.15223338779956427
        },
        "sensitivity": {
          "mean_delta": -0.041187575805516005,
          "ci_low": -0.07769255919422034,
          "ci_high": -0.008870959236407749
        },
        

In [4]:
selected = lower_back['decision']['selected_method']
if selected != 'erm':
    print(f'{selected} passed the lower-back gate; starting the predeclared three-channel confirmation.')
    three_channel = run(ROOT, BenchmarkConfig(representation='three_channel_acceleration'))
    display(three_channel['summary'].sort_values(['method', 'held_out_source']).round(4))
    print(json.dumps(three_channel['decision'], indent=2))
else:
    three_channel = None
    print('No candidate passed the lower-back gate. Three-channel confirmation correctly skipped.')


ensemble_all passed the lower-back gate; starting the predeclared three-channel confirmation.
Device: cuda | NVIDIA GeForce RTX 5060 Laptop GPU
Representation: three_channel_acceleration | channels=[0, 1, 2]
source                label  
felius_2024           healthy     34
                      stroke     129
sint_maartenskliniek  healthy     20
                      stroke      10
voisard_2025          healthy     72
                      stroke      49
Name: participants, dtype: int64


C:\Users\frank\Documents\MR-ICT Review Paper\src\models\evidence_gated_domain_generalization.py:410: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  validation_history = history_frame.loc[history_frame.get("validation", False).fillna(False)].copy()


Tuned felius_2024 erm: 16 epochs


C:\Users\frank\Documents\MR-ICT Review Paper\src\models\evidence_gated_domain_generalization.py:410: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  validation_history = history_frame.loc[history_frame.get("validation", False).fillna(False)].copy()


Tuned felius_2024 coral: 8 epochs


...

Complete voisard_2025 seed=515 method=coral


Complete voisard_2025 seed=515 method=ermpp_style


            method      held_out_source  auroc  brier  balanced_accuracy  specificity  sensitivity  false_positives
             coral          felius_2024 0.8808 0.2562             0.7299       0.8706       0.5891              4.4
             coral sint_maartenskliniek 0.8690 0.1219             0.8100       0.9600       0.6600              0.8
             coral         voisard_2025 0.9420 0.1228             0.8362       0.7500       0.9224             18.0
      ensemble_all          felius_2024 0.8870 0.2222             0.7493       0.8412       0.6574              5.4
      ensemble_all sint_maartenskliniek 0.8830 0.1091             0.8100       0.9200       0.7000              1.6
      ensemble_all         voisard_2025 0.9361 0.1237             0.8397       0.7611       0.9184             17.2
ensemble_erm_coral          felius_2024 0.8891 0.2409             0.7519       0.8882       0.6155              3.8
ensemble_erm_coral sint_maartenskliniek 0.8710 0.1202             0.8150

,method,held_out_source,auroc,brier,balanced_accuracy,specificity,sensitivity,false_positives
0,coral,felius_2024,0.8808,0.2562,0.7299,0.8706,0.5891,4.4
1,coral,sint_maartenskliniek,0.8690,0.1219,0.8100,0.9600,0.6600,0.8
2,coral,voisard_2025,0.9420,0.1228,0.8362,0.7500,0.9224,18.0
3,ensemble_all,felius_2024,0.8870,0.2222,0.7493,0.8412,0.6574,5.4
4,ensemble_all,sint_maartenskliniek,0.8830,0.1091,0.8100,0.9200,0.7000,1.6
5,ensemble_all,voisard_2025,0.9361,0.1237,0.8397,0.7611,0.9184,17.2
6,ensemble_erm_coral,felius_2024,0.8891,0.2409,0.7519,0.8882,0.6155,3.8
7,ensemble_erm_coral,sint_maartenskliniek,0.8710,0.1202,0.8150,0.9500,0.6800,1.0
8,ensemble_erm_coral,voisard_2025,0.9405,0.1315,0.8415,0.7361,0.9469,19.0
9,ensemble_erm_ermpp,felius_2024,0.8879,0.2109,0.7662,0.8471,0.6853,5.2


{
  "selected_method": "ensemble_erm_ermpp",
  "candidate_decisions": {
    "coral": {
      "accepted": false,
      "noninferior": false,
      "material_gain": false,
      "worst_source_deltas": {
        "balanced_accuracy_delta": -0.031144550843593355,
        "specificity_delta": 0.05555555555555558,
        "sensitivity_delta": -0.03875968992248069,
        "auroc_delta": -0.0010000000000000009,
        "mean_brier_delta": -0.0005505096165737688
      },
      "paired_bootstrap": {
        "balanced_accuracy": {
          "mean_delta": -0.003843346123979326,
          "ci_low": -0.029131076849344434,
          "ci_high": 0.020840084806532412
        },
        "specificity": {
          "mean_delta": 0.017342047930283214,
          "ci_low": -0.01695261437908497,
          "ci_high": 0.06266966230936817
        },
        "sensitivity": {
          "mean_delta": -0.025028740178241846,
          "ci_low": -0.07108790803142963,
          "ci_high": 0.012328719084533028
        },

In [5]:
decision = lower_back['decision']
print('FINAL DEVELOPMENT DECISION')
print('Selected method:', decision['selected_method'])
print('Selection scope:', decision['selection_scope'])
print('Frozen cohorts loaded:', decision['frozen_cohorts_loaded'])
print('Interpretation: this is a development-domain robustness decision, not a claim of clinical readiness.')


FINAL DEVELOPMENT DECISION
Selected method: ensemble_all
Selection scope: development-only repeated leave-one-source-out
Frozen cohorts loaded: False
Interpretation: this is a development-domain robustness decision, not a claim of clinical readiness.
